# RAGAS Evaluation — MSADS RAG Agent

End-to-end quality evaluation using RAGAS metrics:
- **Faithfulness** — answer supported by context, no hallucination
- **AnswerRelevancy** — answer addresses the question
- **ContextPrecision** — fraction of retrieved context that is relevant
- **ContextRecall** — retrieved context covers the reference answer

LLM: `qwen3:8b` via Ollama  
Embeddings: `BAAI/bge-small-en-v1.5`

## 1. Environment Check

In [3]:
import sys, json, urllib.request
from pathlib import Path

# --- ragas version ---
try:
    import ragas
    print(f"ragas version: {ragas.__version__}")
except ImportError:
    print("ragas not installed — run: pip install 'ragas>=0.2' langchain-ollama langchain-community")

# --- Ollama connectivity ---
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5) as r:
        tags = json.loads(r.read())
    models = [m["name"] for m in tags.get("models", [])]
    print(f"Ollama available. Models: {models}")
except Exception as e:
    print(f"Ollama not available: {e}")

# --- index files ---
INDEX_DIR = Path("../index")
for f in ["chunks.json", "knowledge_graph.json", "bm25.pkl", "index_meta.json"]:
    p = INDEX_DIR / f
    print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

# --- eval_queries.json ---
QUERIES_PATH = Path("../eval_queries.json")
print(f"{'OK' if QUERIES_PATH.exists() else 'MISSING'}: {QUERIES_PATH}")

ragas version: 0.4.3
Ollama available. Models: ['qwen3:8b']
OK: ..\index\chunks.json
OK: ..\index\knowledge_graph.json
OK: ..\index\bm25.pkl
OK: ..\index\index_meta.json
OK: ..\eval_queries.json


## 2. Load eval_queries.json

In [4]:
import json
from pathlib import Path

QUERIES_PATH = Path("../eval_queries.json")
eval_queries = json.loads(QUERIES_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(eval_queries)} evaluation queries.")
print("Sample:", json.dumps(eval_queries[0], indent=2, ensure_ascii=False))

Loaded 30 evaluation queries.
Sample: {
  "qid": "C1-Q01",
  "category": "admission",
  "sub_type": "single_fact",
  "difficulty": "easy",
  "question": "How many letters of recommendation do I need for the MS in Applied Data Science?",
  "ground_truth_answer": "The program requires two letters of recommendation. They strongly recommend that at least one come from a direct manager, supervisor, or internship supervisor who can speak to your professional skills. Letters from family members, friends, or peers are not accepted.",
  "gold_urls": [
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/"
  ],
  "gold_section_hint": "How to Apply — Letters of Recommendation",
  "expected_behavior": "direct_answer",
  "notes": "Core admission fact, low ambiguity."
}


## 3. Run Agent — Collect Answers and Contexts

In [5]:
import sys, json
from pathlib import Path

# Add project root to path so agent/ and grag/ are importable
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)  # work relative to project root

from agent import agent_loop, answer_generator
from agent.ollama_client import OllamaClient
from agent.tools import AgentTools

tools = AgentTools(index_dir="index")
client = OllamaClient()
print(f"Ollama available: {client.is_available()}")

Ollama available: True


### 3a. 手动测试单条 Query（肉眼看效果）

把 `query` 改成你想问的问题，运行这个 cell，直接看 agent 的回答和证据。  
这个 cell 独立运行，不影响后面的批量 eval。

In [ ]:
# ====== 改这里 ======
query = "What application materials do I need to submit for the MS in Applied Data Science?"
# ====================

memory = agent_loop.run(query=query, history=[], tools=tools, client=client)
response = answer_generator.generate(memory, client)

# --- 运行摘要 ---
print(f"stop_reason : {memory.stop_reason}")
print(f"tool_calls  : {len(memory.tool_calls)}")
print(f"evidence    : {len(memory.evidence_container)} items")

# --- 改写后的 query ---
print("\nRewritten queries:")
for rq in memory.rewritten_queries:
    print(f"  [{rq.get('id','')}] {rq.get('query','')}")

# --- 每条 tool call ---
print("\nTool calls:")
for tc in memory.tool_calls:
    print(f"  step {tc.step}: {tc.tool_name}  args={tc.arguments}")

# --- 保留的 evidence ---
print("\nEvidence kept:")
for ev in memory.evidence_container:
    path = " > ".join(ev.path or [])
    print(f"  [{ev.evidence_id}] {ev.page_title}")
    print(f"           path : {path}")
    print(f"           url  : {ev.source_url}")
    print(f"           why  : {ev.why_kept}")
    print()

# --- 最终回答 ---
print("=" * 60)
print("ANSWER")
print("=" * 60)
print(response.answer)
print()
print("CITATIONS")
for c in response.citations:
    print(f"  [{c.index}] {c.title}")
    print(f"       {c.source_url}")
    print(f"       snippet: {c.snippet[:120]}")

### 3b. 批量运行所有 eval_queries（用于 RAGAS）

In [ ]:
from tqdm.auto import tqdm

CACHE_PATH = Path("agent_run_cache.json")

# Load cache if exists
if CACHE_PATH.exists():
    cache = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cache with {len(cache)} entries.")
else:
    cache = {}

results = []

for row in tqdm(eval_queries, desc="Running agent"):
    qid = row["qid"]
    question = row["question"]
    ground_truth = row.get("ground_truth_answer", "")

    if qid in cache:
        results.append(cache[qid])
        continue

    # Run agent
    memory = agent_loop.run(
        query=question,
        history=[],
        tools=tools,
        client=client,
    )
    response = answer_generator.generate(memory, client)

    contexts = [ev.text for ev in memory.evidence_container]

    entry = {
        "qid": qid,
        "question": question,
        "agent_answer": response.answer,
        "contexts": contexts,
        "ground_truth_answer": ground_truth,
        "stop_reason": memory.stop_reason,
        "tool_call_count": len(memory.tool_calls),
    }
    cache[qid] = entry
    results.append(entry)

    # Save cache incrementally
    CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nCollected {len(results)} results.")

## 4. Build RAGAS EvaluationDataset

In [ ]:
from ragas import EvaluationDataset, SingleTurnSample

samples = [
    SingleTurnSample(
        user_input=row["question"],
        response=row["agent_answer"],
        retrieved_contexts=row["contexts"],
        reference=row["ground_truth_answer"],
    )
    for row in results
    if row["contexts"]  # skip empty-evidence runs for metric stability
]

dataset = EvaluationDataset(samples=samples)
print(f"Dataset: {len(samples)} samples")

## 5. Configure RAGAS LLM + Embeddings

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

# qwen3:8b thinking mode may interfere with RAGAS JSON parsing
# Use /no_think in system prompt or stop token to disable
ragas_llm = LangchainLLMWrapper(
    ChatOllama(
        model="qwen3:8b",
        base_url="http://localhost:11434",
        # Disable thinking mode for reliable RAGAS parsing
        model_kwargs={"options": {"stop": ["</think>"]}},
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

print("RAGAS LLM and embeddings configured.")

## 6. Run RAGAS evaluate()

Estimated: ~30 questions × 4 metrics × ~2 LLM calls ≈ 240 Ollama calls (~20–40 min locally)

In [ ]:
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

result = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

df = result.to_pandas()
print(df.head())

## 7. Print Metric Means

In [ ]:
metrics_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
available = [c for c in metrics_cols if c in df.columns]
print("\n=== RAGAS Metric Means ===")
print(df[available].mean().to_string())

## 8. Low-Score Sample Analysis

In [ ]:
import pandas as pd

for metric in available:
    print(f"\n=== Lowest 5 by {metric} ===")
    low = df.nsmallest(5, metric)[["user_input", "response", metric]]
    for _, row in low.iterrows():
        print(f"  Score={row[metric]:.3f}")
        print(f"  Q: {str(row['user_input'])[:120]}")
        print(f"  A: {str(row['response'])[:200]}")
        print()

## 9. Export Results

In [ ]:
OUT_PATH = Path("ragas_results.csv")
df.to_csv(OUT_PATH, index=False)
print(f"Results saved to {OUT_PATH.resolve()}")